# 三模型净力矩读出与数值修准

在同一数据协议下回答：将外部读出的数值修到真实净力矩后，模型答案会不会随之改善？

[实验设置与运行说明](../../../docs/experiment_notes/03_section44_readout_correction/62dca007edfe57b5.md)


In [1]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')

IMPLEMENTATION_SHA256 = '0915231ffc7f3cc8bcce169946ef4ba98bee1f851221a0c05b94041bb1f102c8'


Initialize progress display:   0%|          | 0/1 [00:00<?, ?cell/s]

Cell progress helper ready: 17 tracked cells.


In [2]:
%%cell_progress 1/15 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


Mount Drive and locate VisualTorque:   0%|          | 0/15 [00:00<?, ?cell/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/VisualTorque


In [3]:
%%cell_progress 2/15 Install and verify the multimodal environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


Install and verify the multimodal environment:   7%|6         | 1/15 [00:00<?, ?cell/s]

torch: 2.11.0+cu128 | CUDA: 12.8
torchvision: 0.26.0+cu128
Toolkit source: /content/drive/MyDrive/VisualTorque/toolkit/src/visual_torque_toolkit/__init__.py


In [4]:
%%cell_progress 3/15 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


Verify A100 and Hugging Face access:  13%|#3        | 2/15 [00:00<?, ?cell/s]

NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA memory (GB): 95.0
Hugging Face login skipped: Secret HF_TOKEN does not exist.


In [5]:
%%cell_progress 4/15 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [
                member for member in bundle.infolist()
                if not member.is_dir()
            ]
            root = DATASET_DIR.resolve()
            for member in tqdm(
                members, desc='Extracting Dataset v1', unit='file'
            ):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


Prepare Dataset v1 on local SSD:  20%|##        | 3/15 [00:00<?, ?cell/s]

Dataset: /content/visualtorque_dataset_v1


## 1. 选择模型并冻结运行参数


In [6]:
%%cell_progress 5/15 Run analysis
cell_stage('Locking model, numerical targets, controls, and output identity')
import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import transformers

MODEL_KEY = 'qwen27b'  # qwen9b / gemma12b / qwen27b; one fresh runtime per model
MODEL_SPECS = {
    'qwen9b': ('Qwen/Qwen3.5-9B', 'Qwen__Qwen3.5-9B', 32,
               'c202236235762e1c871ad0ccb60c8ee5ba337b9a', 13, .01),
    'gemma12b': ('google/gemma-4-12B-it', 'google__gemma-4-12B-it', 48, None, 7, 100.),
    'qwen27b': ('Qwen/Qwen3.5-27B', 'Qwen__Qwen3.5-27B__373612cb', 64,
                '373612cb4f9288fcf550b526d86d85febedd9801', None, None),
}
MODEL_ID, MODEL_SLUG, EXPECTED_LAYERS, MODEL_REVISION, EXPECTED_LAYER, EXPECTED_ALPHA = MODEL_SPECS[MODEL_KEY]
MODEL_DTYPE = 'bfloat16'
BATCH_SIZE = 1
SEED = 20260820  # Exact original 304-scene / 72-control / mapping selection.
CALIBRATION_STEPS = 32
FIXED_DOSE_CALIBRATION_STEPS = 12  # Original v2 intervention.
READOUT_TOLERANCE = 0.05
NORM_TOLERANCE = 0.02
BALANCE_THRESHOLD = 0.5  # Same original outcome decoder; not the write-in tolerance.
PARITY_TOLERANCE = 0.001
RANDOM_SEEDS = (0, 1, 2, 3)
RIDGE_ALPHAS = (.01, .1, 1., 10., 100., 1000.)
DOSE_MULTIPLIERS = (-2., -1., 0., 1., 2.)
SCENE_CAPS = {'distance': 54, 'conflict_distance': 42, 'conflict_balance': 12,
              'balance': 6, 'weight': 40, 'conflict_weight': 40,
              'compatible_weight_distance': 40, 'multi_stack': 70}
SHARED_ROOT = PROJECT_ROOT / 'results' / 'cross_model_readout_correction_v2'
OUTPUT_ROOT = SHARED_ROOT / 'models' / MODEL_SLUG
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
DISCOVERY_ROOT = PROJECT_ROOT / 'results' / MODEL_SLUG / 'activation_discovery_v1'
CONFIRMATORY_ROOT = PROJECT_ROOT / 'results' / MODEL_SLUG / 'activation_confirmatory_v1'
CAUSAL_ROOT = PROJECT_ROOT / 'results' / MODEL_SLUG / 'causal_torque_intervention_v2'
ACTIVATION_MANIFEST = PROJECT_ROOT / 'results' / 'behavior_comparison_v3' / 'activation_pilot_manifest.jsonl'
for folder in (OUTPUT_ROOT, ANALYSIS_DIR, SHARED_ROOT / 'cross_model_analysis'):
    folder.mkdir(parents=True, exist_ok=True)

def json_plain(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)

def canonical(value):
    return json.loads(json.dumps(value, sort_keys=True, default=json_plain, allow_nan=False))

def identity_hash(value):
    return hashlib.sha256(json.dumps(canonical(value), sort_keys=True).encode()).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(canonical(value), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def read_jsonl(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]

def lock_json(path, value):
    value = canonical(value)
    if Path(path).exists():
        assert read_json(path) == value, f'Identity differs; do not mix runs: {path}'
    else:
        write_json(path, value)
    return identity_hash(value)

discovery_config_path = DISCOVERY_ROOT / 'capture_config.json'
discovery_selection_path = DISCOVERY_ROOT / 'probe_results' / 'discovery_selection.json'
discovery_activation_path = DISCOVERY_ROOT / 'scene_mean_activations.pt'
if MODEL_REVISION is None:
    assert discovery_config_path.is_file(), f'Missing original pinned checkpoint: {discovery_config_path}'
    MODEL_REVISION = read_json(discovery_config_path)['identity']['resolved_revision']
assert re.fullmatch(r'[0-9a-f]{40}', MODEL_REVISION)
protocol = dict(schema='readout-correction-1.1', model_id=MODEL_ID, revision=MODEL_REVISION,
    implementation=IMPLEMENTATION_SHA256, dtype=MODEL_DTYPE, batch_size=BATCH_SIZE,
    readout_tolerance=READOUT_TOLERANCE, balance_threshold=BALANCE_THRESHOLD,
    norm_tolerance=NORM_TOLERANCE, random_seeds=RANDOM_SEEDS, calibration_steps=CALIBRATION_STEPS,
    fixed_dose_steps=FIXED_DOSE_CALIBRATION_STEPS, ridge_alphas=RIDGE_ALPHAS,
    dose_multipliers=DOSE_MULTIPLIERS, selection_seed=SEED, scene_caps=SCENE_CAPS,
    primary_estimand='paired model accuracy change, oracle correction minus zero replay, all 304 scenes',
    controls='raw random and standardized-orthogonal; match actual raw residual norm; 4 seeds averaged within scene',
    oracle_uses_true_torque=True, bootstrap_unit='scene', bootstrap_replicates=10000,
    accuracy_sesoi=.05, reuse_policy='verified original per-example records only; paired intervention baseline always fresh')
PROTOCOL_SHA = lock_json(OUTPUT_ROOT / 'protocol.json', protocol)
print('Model:', MODEL_ID, MODEL_REVISION)
print('Output:', OUTPUT_ROOT)


Run analysis:  27%|##6       | 4/15 [00:00<?, ?cell/s]

Model: Qwen/Qwen3.5-27B 373612cb4f9288fcf550b526d86d85febedd9801
Output: /content/drive/MyDrive/VisualTorque/results/cross_model_readout_correction_v2/models/Qwen__Qwen3.5-27B__373612cb


## 2. 读取原训练/开发清单与模型


In [7]:
%%cell_progress 6/15 Run analysis
cell_stage('Loading the 254 discovery scenes; test outcomes are not used for selection')
from visual_torque_toolkit.behavior import BehaviorRunConfig, PromptCondition, load_behavior_examples
assert ACTIVATION_MANIFEST.is_file(), f'Missing original 254-scene discovery manifest: {ACTIVATION_MANIFEST}'
discovery_manifest = read_jsonl(ACTIVATION_MANIFEST)
assert len(discovery_manifest) == 254
fit_ids = {str(row['scene_id']) for row in discovery_manifest}
assert len(fit_ids) == 254
assert {row['split'] for row in discovery_manifest} <= {'probe_train', 'probe_dev'}
def example_config(splits):
    return BehaviorRunConfig(dataset_dir=DATASET_DIR, output_dir=Path('/content/unused_e1'),
        model_id=MODEL_ID, model_revision=MODEL_REVISION, conditions=(PromptCondition.IMAGE_RULE,),
        question_kinds=('outcome',), splits=tuple(splits), batch_size=1, generation_every=0, seed=SEED)
fit_examples = [e for e in load_behavior_examples(example_config(('probe_train', 'probe_dev')))
                if str(e.scene_id) in fit_ids]
assert len(fit_examples) == 254 * 6
fit_by_scene = defaultdict(list)
for example in fit_examples:
    fit_by_scene[str(example.scene_id)].append(example)
assert all(len(v) == 6 and len({e.variant for e in v}) == 6 for v in fit_by_scene.values())
for row in discovery_manifest:
    assert {e.split for e in fit_by_scene[str(row['scene_id'])]} == {row['split']}

def example_identity(example):
    return dict(example_id=example.example_id, scene_id=str(example.scene_id), split=example.split,
        variant=example.variant, prompt=example.prompt, labels=list(example.labels),
        label_to_semantic={str(k): str(v) for k,v in example.label_to_semantic.items()},
        image_sha256=sha256_file(example.image_path), net_torque=int(example.net_torque),
        gold_semantic=str(example.gold_semantic), problem_type=example.problem_type)

toolkit_identity = {str(p.relative_to(TOOLKIT_ROOT)): sha256_file(p)
                    for p in sorted((TOOLKIT_ROOT / 'src' / 'visual_torque_toolkit').rglob('*.py'))}
runtime_identity = dict(torch=torch.__version__, transformers=transformers.__version__,
    toolkit=toolkit_identity, dataset=example_config(('probe_train','probe_dev')).dataset_fingerprint(),
    manifest=sha256_file(ACTIVATION_MANIFEST))
RUNTIME_SHA = lock_json(OUTPUT_ROOT / 'runtime_identity.json', runtime_identity)

import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
adapter = HuggingFaceMultimodalAdapter(MODEL_ID, revision=MODEL_REVISION, dtype=MODEL_DTYPE, device_map='auto')
assert adapter.metadata['resolved_revision'] == MODEL_REVISION
DECODER_LAYERS = None
for candidate in ('model.language_model.layers', 'model.text_model.layers', 'language_model.model.layers',
                  'language_model.layers', 'model.model.layers', 'model.layers'):
    module = adapter.model
    try:
        for component in candidate.split('.'):
            module = getattr(module, component)
    except AttributeError:
        continue
    if isinstance(module, torch.nn.ModuleList) and len(module) == EXPECTED_LAYERS:
        DECODER_LAYERS, LAYER_PATH = module, candidate
        break
assert DECODER_LAYERS is not None, 'Cannot find the expected decoder layers'
assert not any(v == 'disk' for v in getattr(adapter.model, 'hf_device_map', {}).values()), 'Use enough memory; disk offload is unsuitable here'
print('Decoder layers:', len(DECODER_LAYERS), '| native checkpoint:', MODEL_REVISION)


Run analysis:  33%|###3      | 5/15 [00:00<?, ?cell/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Decoder layers: 64 | native checkpoint: 373612cb4f9288fcf550b526d86d85febedd9801


## 3. 定义单样本前向、原生精度写入与断点保存


In [8]:
%%cell_progress 7/15 Run analysis
cell_stage('Defining exact final-prompt capture and scoring')
def output_hidden(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and torch.is_tensor(output[0]):
        hidden = output[0]
    else:
        raise TypeError(f'Unsupported decoder output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_hidden(output, value):
    if torch.is_tensor(output):
        return value
    if isinstance(output, tuple):
        return (value,) + output[1:]
    if isinstance(output, list):
        return [value] + output[1:]
    raise TypeError(type(output).__name__)

def forward_one(example, layers=(), edit=None):
    plan = adapter._candidate_plan(example)
    assert plan is not None, 'A/B/C must have the existing single-token scoring plan'
    prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(adapter._to_device(adapter._template(
        [example.messages()], add_generation_prompt=True)), prefix)
    assert bool(inputs['attention_mask'][0, -1]), 'Final position is padded'
    captured, audits, handles = {}, {}, []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden = output_hidden(output)
            assert hidden.dtype == torch.bfloat16, hidden.dtype
            pre = hidden[0, -1].detach().float().cpu().numpy().copy()
            captured[layer] = pre
            if edit is None:
                return None
            assert len(layers) == 1
            cast = lambda x: torch.as_tensor(np.asarray(x).copy(), dtype=hidden.dtype).float().numpy()
            post, audit = edit(pre, cast)
            modified = hidden.clone()
            modified[0, -1] = torch.as_tensor(post, dtype=hidden.dtype, device=hidden.device)
            actual = modified[0, -1].detach().float().cpu().numpy()
            audit.update(actual_delta_norm=float(np.linalg.norm(actual - pre)),
                         standardized_delta_norm=float(np.linalg.norm((actual-pre)/PROBE_SCALE)),
                         decoded_pre=decode(pre), decoded_post=decode(actual),
                         hidden_norm=float(np.linalg.norm(pre)))
            audits.update(audit)
            return replace_hidden(output, modified)
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**inputs, use_cache=False)
            scores = torch.log_softmax(output.logits[0, -1].float(), dim=-1)
            label_scores = {label: float(scores[token_by_label[label]].cpu()) for label in example.labels}
    finally:
        for handle in handles:
            handle.remove()
    assert set(captured) == set(layers)
    predicted_label = max(example.labels, key=lambda label: label_scores[label])
    semantic_scores = {str(example.label_to_semantic[label]): value for label,value in label_scores.items()}
    result = dict(example_id=example.example_id, scene_id=str(example.scene_id), variant=example.variant,
        split=example.split, problem_type=example.problem_type, net_torque=int(example.net_torque),
        gold_semantic=str(example.gold_semantic), predicted_semantic=str(example.label_to_semantic[predicted_label]),
        correct=str(example.label_to_semantic[predicted_label]) == str(example.gold_semantic),
        label_to_semantic={str(k):str(v) for k,v in example.label_to_semantic.items()},
        candidate_logprobs=label_scores, semantic_logprobs=semantic_scores, **audits)
    return result, captured

def cached_scene(stage, scene_id, identity, compute):
    directory = OUTPUT_ROOT / stage
    directory.mkdir(parents=True, exist_ok=True)
    filename = hashlib.sha256(str(scene_id).encode()).hexdigest()[:24] + '.json'
    path = directory / filename
    expected = identity_hash(dict(protocol=PROTOCOL_SHA, runtime=RUNTIME_SHA, stage=stage, identity=identity))
    if path.is_file():
        payload = read_json(path)
        assert payload['identity'] == expected and payload['scene_id'] == str(scene_id), path
        return payload['result']
    result = compute()
    write_json(path, dict(identity=expected, scene_id=str(scene_id), result=result))
    return canonical(result)


Run analysis:  40%|####      | 6/15 [00:00<?, ?cell/s]

## 4. 复用 ridge；27B 缺少归档时在原开发清单选层


In [9]:
%%cell_progress 8/15 Run analysis
cell_stage('Fitting the readout without test outcomes')
archive_available = all(p.is_file() for p in (discovery_config_path, discovery_selection_path, discovery_activation_path))
if EXPECTED_LAYER is not None:
    assert archive_available, 'The small-model experiment requires its original discovery artifacts; do not refit on a different sample'
if archive_available:
    di = read_json(discovery_config_path)['identity']
    selection = read_json(discovery_selection_path)
    assert di['model_id'] == MODEL_ID and di['resolved_revision'] == MODEL_REVISION
    assert di['dataset_fingerprint'] == runtime_identity['dataset']
    assert di['activation_manifest_sha256'] == sha256_file(ACTIVATION_MANIFEST)
    assert selection['probe_test_used'] is False
    assert selection['selection_data'] == ['probe_train', 'probe_dev']
    selected = selection['candidate_net_torque_layer']
    SELECTED_LAYER = int(selected['layer_index'])
    FROZEN_ALPHA = float(selected['selected_alpha'])
    if EXPECTED_LAYER is not None:
        assert (SELECTED_LAYER, FROZEN_ALPHA) == (EXPECTED_LAYER, EXPECTED_ALPHA)
    payload = torch.load(discovery_activation_path, map_location='cpu', weights_only=False)
    meta = pd.DataFrame(payload['metadata'])
    assert meta['scene_id'].astype(str).is_unique and set(meta['scene_id'].astype(str)) == fit_ids
    assert set(meta['split']) <= {'probe_train', 'probe_dev'}
    for row in meta.to_dict('records'):
        ex = fit_by_scene[str(row['scene_id'])][0]
        assert row['split'] == ex.split and int(row['net_torque']) == int(ex.net_torque)
    li = [int(x) for x in payload['layer_indices']].index(SELECTED_LAYER)
    X_fit_raw = payload['activations'][:, li].float().numpy()
    y_fit = meta['net_torque'].to_numpy(dtype=float)
    fit_provenance = dict(mode='original_discovery_archive', files={str(p):sha256_file(p)
        for p in (discovery_config_path, discovery_selection_path, discovery_activation_path)})
else:
    # Same eight-point grid and original 254-scene pilot as the two small models.
    grid = tuple(sorted({int(round(x)) for x in np.linspace(0, EXPECTED_LAYERS-1, 8)}))
    rows, activations = [], []
    for scene_id in tqdm(sorted(fit_ids), desc='Qwen-27B discovery scene means', unit='scene'):
        examples = sorted(fit_by_scene[scene_id], key=lambda e:e.variant)
        def capture_fit(examples=examples):
            states = []
            for example in examples:
                _, captured = forward_one(example, grid)
                states.append(np.stack([captured[layer] for layer in grid]).astype(np.float16).astype(np.float32))
            # Match original float16 scene-mean storage before probe fitting.
            return {'mean': np.mean(states, axis=0).astype(np.float16).astype(np.float32).tolist()}
        result = cached_scene('discovery_shards', scene_id,
            dict(layers=grid, examples=[example_identity(e) for e in examples]), capture_fit)
        activations.append(np.asarray(result['mean'], dtype=np.float32))
        rows.append(dict(scene_id=scene_id, split=examples[0].split, net_torque=int(examples[0].net_torque)))
    X_all = np.stack(activations)
    meta = pd.DataFrame(rows)
    y_fit = meta['net_torque'].to_numpy(dtype=float)
    train, dev = meta['split'].eq('probe_train').to_numpy(), meta['split'].eq('probe_dev').to_numpy()
    assert train.sum() > 1 and dev.sum() > 1
    selection_rows = []
    for pos, layer in enumerate(grid):
        scaler = StandardScaler().fit(X_all[train, pos])
        for alpha in RIDGE_ALPHAS:
            estimator = Ridge(alpha=alpha, solver='lsqr', max_iter=5000).fit(scaler.transform(X_all[train, pos]), y_fit[train])
            score = float(r2_score(y_fit[dev], estimator.predict(scaler.transform(X_all[dev, pos]))))
            assert np.isfinite(score)
            selection_rows.append(dict(layer=layer, alpha=alpha, dev_r2=score))
    best = max(selection_rows, key=lambda row:(row['dev_r2'], -row['alpha'], -row['layer']))
    SELECTED_LAYER, FROZEN_ALPHA = int(best['layer']), float(best['alpha'])
    X_fit_raw = X_all[:, grid.index(SELECTED_LAYER)]
    fit_provenance = dict(mode='same_254_scene_ridge_discovery', manifest_sha=sha256_file(ACTIVATION_MANIFEST), grid=grid)
    write_json(OUTPUT_ROOT / 'development_selection.json', dict(all_candidates=selection_rows, selected=best))
scaler = StandardScaler()
X_fit = scaler.fit_transform(X_fit_raw)
probe = Ridge(alpha=FROZEN_ALPHA, solver='lsqr', max_iter=5000).fit(X_fit, y_fit)
PROBE_MEAN = np.asarray(scaler.mean_, dtype=np.float32)
PROBE_SCALE = np.asarray(scaler.scale_, dtype=np.float32)
PROBE_COEF = np.asarray(probe.coef_, dtype=np.float32)
PROBE_INTERCEPT = float(probe.intercept_)
TORQUE_SHIFT_UNIT = .5 * float(np.std(y_fit, ddof=0))
assert TORQUE_SHIFT_UNIT > 0
probe_artifact = dict(layer=SELECTED_LAYER, alpha=FROZEN_ALPHA, mean=PROBE_MEAN, scale=PROBE_SCALE,
    coef=PROBE_COEF, intercept=PROBE_INTERCEPT, torque_shift_unit=TORQUE_SHIFT_UNIT,
    fit_scene_ids=meta['scene_id'].astype(str).tolist(), fit_provenance=fit_provenance)
PROBE_SHA = lock_json(OUTPUT_ROOT / 'frozen_probe.json', probe_artifact)
print('Frozen layer / ridge alpha:', SELECTED_LAYER, FROZEN_ALPHA)
print('Fit scenes:', len(y_fit), '| five-dose unit:', TORQUE_SHIFT_UNIT)


Run analysis:  47%|####6     | 7/15 [00:00<?, ?cell/s]

Qwen-27B discovery scene means:   0%|          | 0/254 [00:00<?, ?scene/s]

Frozen layer / ridge alpha: 27 0.1
Fit scenes: 254 | five-dose unit: 4.4041517677193625


## 5. 数值修准、原五剂量与匹配随机变化


In [10]:
%%cell_progress 9/15 Run analysis
cell_stage('Defining the numerical correction and finite-precision controls')
"""Pure numerical kernels embedded in the E1 notebook; cast is supplied by the runtime."""
import hashlib
import numpy as np


def readout_value(hidden, mean, scale, coef, intercept):
    return float(((np.asarray(hidden, dtype=np.float64) - mean) / scale) @ coef + intercept)


def numerical_correction(hidden, target, mean, scale, coef, intercept, cast,
                         tolerance=0.05, steps=32):
    """Calibrate only the external readout; downstream always receives native dtype.

    The initial edit minimizes standardized-space norm. Subsequent quantization
    calibration need not be the globally smallest native-dtype perturbation.
    """
    h = np.asarray(hidden, dtype=np.float64)
    a = np.asarray(coef, dtype=np.float64)
    scale = np.asarray(scale, dtype=np.float64)
    assert np.all(scale > 0) and float(a @ a) > 0 and steps > 0
    initial = readout_value(h, mean, scale, a, intercept)
    unit = scale * a / float(a @ a)
    planned = (target - initial) * unit
    # Preserve an exact zero edit for already-in-tolerance examples.
    best = h.copy()
    error = abs(initial - target)
    if error <= tolerance:
        return best, dict(planned_delta_norm=0.0, residual=error, achieved=True,
                          iterations=0, originally_within_tolerance=True)
    work = planned.copy()
    for iteration in range(steps):
        candidate = np.asarray(cast(h + work), dtype=np.float64)
        value = readout_value(candidate, mean, scale, a, intercept)
        candidate_error = abs(target - value)
        if candidate_error < error:
            best, error = candidate.copy(), candidate_error
        if error <= tolerance:
            break
        work += (target - value) * unit
    return best, dict(planned_delta_norm=float(np.linalg.norm(planned)),
                      residual=float(error), achieved=bool(error <= tolerance),
                      iterations=iteration + 1, originally_within_tolerance=False)


def deterministic_direction(shape, key, scale, coef, orthogonal=False):
    seed = int(hashlib.sha256(str(key).encode()).hexdigest()[:16], 16)
    z = np.random.default_rng(seed).normal(size=shape)
    if orthogonal:
        z -= coef * float(z @ coef) / float(coef @ coef)
    raw = z * scale
    assert np.linalg.norm(raw) > 0
    return raw / np.linalg.norm(raw)


def match_native_norm(hidden, direction, target_norm, cast, tolerance=0.02, steps=48):
    """Bounded amplitude search, matching actual raw residual norm after casting.

    A discrete native dtype can make exact norm matching impossible. Return the
    best candidate and an explicit failure flag; never silently call it matched.
    """
    h = np.asarray(hidden, dtype=np.float64)
    direction = np.asarray(direction, dtype=np.float64)
    assert target_norm >= 0 and np.linalg.norm(direction) > 0
    direction = direction / np.linalg.norm(direction)
    if target_norm == 0:
        return h.copy(), dict(norm_error_relative=0.0, norm_matched=True, amplitude=0.0)
    lo, hi = 0.0, max(target_norm, 1e-12)
    for _ in range(32):
        if np.linalg.norm(np.asarray(cast(h + hi * direction)) - h) >= target_norm:
            break
        hi *= 2
    best, best_error, best_amplitude = h.copy(), 1.0, 0.0
    for _ in range(steps):
        amplitude = (lo + hi) / 2
        candidate = np.asarray(cast(h + amplitude * direction), dtype=np.float64)
        norm = float(np.linalg.norm(candidate - h))
        error = abs(norm - target_norm) / target_norm
        if error < best_error:
            best, best_error, best_amplitude = candidate.copy(), error, amplitude
        if error <= tolerance:
            break
        if norm < target_norm:
            lo = amplitude
        else:
            hi = amplitude
    return best, dict(norm_error_relative=float(best_error),
                      norm_matched=bool(best_error <= tolerance), amplitude=float(best_amplitude))


def scene_bootstrap(values, seed=20260913, replicates=10000):
    """Rows must already be independent scene/family means, not mappings/seeds."""
    values = np.asarray(values, dtype=float)
    assert values.ndim == 1 and len(values) and np.isfinite(values).all()
    rng = np.random.default_rng(seed)
    means = []
    for start in range(0, replicates, 250):
        size = min(250, replicates - start)
        means.extend(values[rng.integers(0, len(values), (size, len(values)))].mean(axis=1))
    return dict(mean=float(values.mean()), n=len(values),
                lower=float(np.quantile(means, .025)), upper=float(np.quantile(means, .975)))


def paired_on_ids(left, right, id_column, value_column):
    """Do not silently turn partial overlap or duplicate rows into a paired comparison."""
    assert left[id_column].is_unique and right[id_column].is_unique
    assert set(left[id_column]) == set(right[id_column]), 'Different paired support'
    joined = left[[id_column, value_column]].merge(
        right[[id_column, value_column]], on=id_column, suffixes=('_left', '_right'), validate='one_to_one')
    return joined[value_column + '_left'].to_numpy() - joined[value_column + '_right'].to_numpy()

def decode(hidden):
    return readout_value(hidden, PROBE_MEAN, PROBE_SCALE, PROBE_COEF, PROBE_INTERCEPT)

def probe_outcome(value):
    return 'left' if value > BALANCE_THRESHOLD else 'right' if value < -BALANCE_THRESHOLD else 'balanced'

def oracle_edit(gold):
    def apply(pre, cast):
        candidate, audit = numerical_correction(pre, float(gold), PROBE_MEAN, PROBE_SCALE,
            PROBE_COEF, PROBE_INTERCEPT, cast, READOUT_TOLERANCE, CALIBRATION_STEPS)
        audit.update(condition='oracle', target_torque=float(gold))
        return candidate, audit
    return apply

def zero_edit(pre, cast):
    return pre.copy(), dict(condition='zero', planned_delta_norm=0.)

def fixed_edit(multiplier):
    def apply(pre, cast):
        # Reproduce v2: float32 correction, all 12 steps, keep the smallest realized error.
        coefficient = torch.as_tensor(PROBE_COEF)
        scale = torch.as_tensor(PROBE_SCALE)
        unit = scale * coefficient / torch.dot(coefficient, coefficient)
        h = torch.as_tensor(pre, dtype=torch.float32)
        read = lambda v: ((v.float()-torch.as_tensor(PROBE_MEAN))/scale) @ coefficient + PROBE_INTERCEPT
        before = read(h)
        target_shift = float(multiplier * TORQUE_SHIFT_UNIT)
        work = target_shift * unit
        best, best_error = h.clone(), float('inf')
        for _ in range(FIXED_DOSE_CALIBRATION_STEPS):
            candidate = torch.as_tensor(cast((h+work).numpy()))
            residual = target_shift - (read(candidate)-before)
            error = float(residual.abs())
            if error < best_error:
                best, best_error = candidate.clone(), error
            work += residual * unit
        return best.numpy(), dict(condition=f'fixed_{multiplier:+g}', dose_multiplier=multiplier,
            target_torque=float(before)+target_shift, planned_probe_shift=target_shift,
            residual=best_error, achieved=best_error <= max(.15, .03*abs(target_shift)))
    return apply

def control_edit(scene_id, kind, seed, actual_oracle_norm):
    def apply(pre, cast):
        # Random is isotropic in raw residual space; orthogonal is formed in standardized space.
        scale = PROBE_SCALE if kind == 'orthogonal' else np.ones_like(PROBE_SCALE)
        direction = deterministic_direction(pre.shape, f'{SEED}:{scene_id}:{kind}:{seed}',
            scale, PROBE_COEF, orthogonal=kind == 'orthogonal')
        candidate, audit = match_native_norm(pre, direction, actual_oracle_norm, cast, NORM_TOLERANCE)
        leakage = decode(candidate) - decode(pre)
        audit.update(condition=kind, random_seed=seed, oracle_actual_norm=actual_oracle_norm,
            orthogonal_readout_leakage=leakage if kind == 'orthogonal' else None,
            orthogonal_preserved=abs(leakage) <= READOUT_TOLERANCE if kind == 'orthogonal' else None)
        return candidate, audit
    return apply


Run analysis:  53%|#####3    | 8/15 [00:00<?, ?cell/s]

## 6. 只用开发样本检查写入与零剂量重放


In [11]:
%%cell_progress 10/15 Run analysis
cell_stage('Checking implementation on fixed development examples, before test capture')
dev_examples = sorted((e for e in fit_examples if e.split == 'probe_dev'),
    key=lambda e:hashlib.sha256(f'dev-write:{e.example_id}'.encode()).hexdigest())[:8]
assert len(dev_examples) == 8
dev_rows = []
for example in tqdm(dev_examples, desc='Development write-in checks'):
    plain, _ = forward_one(example)
    zero, _ = forward_one(example, (SELECTED_LAYER,), zero_edit)
    assert max(abs(plain['candidate_logprobs'][k]-zero['candidate_logprobs'][k])
               for k in plain['candidate_logprobs']) < PARITY_TOLERANCE
    assert zero['actual_delta_norm'] == 0
    oracle, _ = forward_one(example, (SELECTED_LAYER,), oracle_edit(example.net_torque))
    assert np.isfinite(oracle['decoded_post']) and np.isfinite(oracle['actual_delta_norm'])
    dev_rows.append(oracle)
    for kind in ('random', 'orthogonal'):
        control, _ = forward_one(example, (SELECTED_LAYER,), control_edit(example.scene_id, kind, 0, oracle['actual_delta_norm']))
        dev_rows.append(control)
write_json(OUTPUT_ROOT / 'development_write_audit.json', dev_rows)
display(pd.DataFrame(dev_rows)[['scene_id','condition','actual_delta_norm','decoded_pre','decoded_post','achieved','norm_matched']])
dev_success = [r['achieved'] for r in dev_rows if r['condition'] == 'oracle']
print('Oracle tolerance attained:', sum(dev_success), '/', len(dev_success))
assert any(dev_success), ('None of the fixed development examples reached the numerical tolerance. '
    'Inspect development_write_audit.json before running test interventions; do not loosen the threshold on test data.')
# Numerical failures are reported, not grounds for choosing a new tolerance using test outcomes.
NUMERIC_LOCK = lock_json(OUTPUT_ROOT / 'numerical_protocol_lock.json', dict(probe_sha=PROBE_SHA,
    tolerance=READOUT_TOLERANCE, steps=CALIBRATION_STEPS, control_norm_tolerance=NORM_TOLERANCE,
    dev_example_ids=[e.example_id for e in dev_examples], dev_oracle_successes=sum(dev_success)))


Run analysis:  60%|######    | 9/15 [00:00<?, ?cell/s]

Development write-in checks:   0%|          | 0/8 [00:00<?, ?it/s]

,scene_id,condition,actual_delta_norm,decoded_pre,decoded_post,achieved,norm_matched
0,single_lu2_ld2_ru2_rd2,oracle,0.007519,0.723927,0.033177,True,NaN
1,single_lu2_ld2_ru2_rd2,random,0.007406,0.723927,0.747855,NaN,True
2,single_lu2_ld2_ru2_rd2,orthogonal,0.007499,0.723927,0.722611,NaN,True
3,multi_c0e457437617298d,oracle,0.028871,1.921287,-2.022391,True,NaN
4,multi_c0e457437617298d,random,0.029324,1.921287,1.965144,NaN,True
5,multi_c0e457437617298d,orthogonal,0.028545,1.921287,1.926478,NaN,True
6,single_lu4_ld2_ru4_rd2,oracle,0.009333,-0.950441,-0.042338,True,NaN
7,single_lu4_ld2_ru4_rd2,random,0.009246,-0.950441,-0.941264,NaN,True
8,single_lu4_ld2_ru4_rd2,orthogonal,0.009194,-0.950441,-0.950405,NaN,True
9,single_lu1_ld3_ru3_rd1,oracle,0.004982,-0.422524,-0.032882,True,NaN


Oracle tolerance attained: 8 / 8


## 7. 锁定原 1,007 场景、304 场景与 72 个对照场景


In [12]:
%%cell_progress 11/15 Run analysis
cell_stage('Reconstructing the exact historical evaluation cohorts')
all_examples = tuple(load_behavior_examples(example_config(('probe_test',))))
test_by_scene = defaultdict(list)
for example in all_examples:
    test_by_scene[str(example.scene_id)].append(example)
assert len(test_by_scene) == 1007 and len(all_examples) == 6042
assert set(test_by_scene).isdisjoint(fit_ids)
assert all(len(v) == 6 and len({e.variant for e in v}) == 6 for v in test_by_scene.values())
scene_by_id = {str(s['scene_id']):s for s in read_jsonl(DATASET_DIR/'scenes.jsonl')}
def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value}'.encode()).hexdigest()
def balanced_take(scenes, cap):
    groups = defaultdict(list)
    for scene in scenes:
        groups[str(scene['derived']['outcome'])].append(scene)
    for values in groups.values():
        values.sort(key=lambda scene:stable_key(scene['scene_id']))
    selected, cursor = [], {k:0 for k in groups}
    while len(selected) < min(cap, len(scenes)):
        for key in sorted(groups):
            if cursor[key] < len(groups[key]) and len(selected) < cap:
                selected.append(groups[key][cursor[key]])
                cursor[key] += 1
    return selected
selected_scenes = []
for problem_type, cap in SCENE_CAPS.items():
    selected_scenes += balanced_take([scene_by_id[s] for s in test_by_scene
                                     if scene_by_id[s]['problem_type'] == problem_type], cap)
selected_examples = []
for scene in sorted(selected_scenes, key=lambda s:s['scene_id']):
    sid = str(scene['scene_id'])
    variant = f"permutation_{int(stable_key('variant:'+sid)[:8],16)%6}"
    selected_examples.append(next(e for e in test_by_scene[sid] if e.variant == variant))
groups = defaultdict(list)
for example in selected_examples:
    groups[example.problem_type].append(example)
for values in groups.values():
    values.sort(key=lambda e:stable_key('control:'+e.scene_id))
control_examples, cursor = [], {k:0 for k in groups}
while len(control_examples) < 72:
    for key in sorted(groups):
        if cursor[key] < len(groups[key]) and len(control_examples) < 72:
            control_examples.append(groups[key][cursor[key]])
            cursor[key] += 1
control_ids = {str(e.scene_id) for e in control_examples}
selection_manifest = [dict(schema_version='causal-selection-1.0', example_id=e.example_id,
    scene_id=str(e.scene_id), variant=e.variant, problem_type=e.problem_type,
    gold_outcome=str(e.gold_semantic), net_torque=int(e.net_torque),
    orthogonal_control_subset=str(e.scene_id) in control_ids) for e in selected_examples]
assert len(selection_manifest) == 304 and len(control_ids) == 72
assert Counter(e.gold_semantic for e in selected_examples) == {'left':132, 'right':131, 'balanced':41}
for spec in MODEL_SPECS.values():
    old_manifest = PROJECT_ROOT/'results'/spec[1]/'causal_torque_intervention_v2'/'selection_manifest.jsonl'
    if old_manifest.is_file():
        assert read_jsonl(old_manifest) == selection_manifest, f'Historical cohort mismatch: {old_manifest}'
COHORT_SHA = lock_json(SHARED_ROOT/'shared_304_manifest.json', selection_manifest)
lock_json(SHARED_ROOT/'shared_1007_scene_ids.json', sorted(test_by_scene))
input_identities = {e.example_id:example_identity(e) for e in tqdm(all_examples, desc='Fingerprinting exact test inputs')}
INPUT_SHA = lock_json(OUTPUT_ROOT/'test_inputs.json', input_identities)
print('Evaluation scenes / selected scenes / control scenes:', len(test_by_scene), len(selected_examples), len(control_ids))


Run analysis:  67%|######6   | 10/15 [00:00<?, ?cell/s]

Fingerprinting exact test inputs:   0%|          | 0/6042 [00:00<?, ?it/s]

Evaluation scenes / selected scenes / control scenes: 1007 304 72


## 8. 核验旧测试状态；不兼容时自动重新采集


In [13]:
%%cell_progress 12/15 Run analysis
# Replace the ENTIRE failing "12/15" cell in the original notebook with this code.
# Keep the current Colab runtime; this cell uses the model and frozen probe already loaded.
# This is a notebook cell, not a standalone Python program.
cell_stage('Auditing archived states; recapturing test states when reuse is not supported')
TEST_REUSE_IMPLEMENTATION_SHA = '29296b854a7bddbf2da6f51826290f448e4ec279eeaa425f226c862ddf5aa48a'
required_globals = ('adapter', 'forward_one', 'decode', 'SELECTED_LAYER', 'test_by_scene',
                    'input_identities', 'INPUT_SHA', 'PROBE_SHA', 'NUMERIC_LOCK',
                    'PROTOCOL_SHA', 'RUNTIME_SHA', 'cached_scene')
missing = [name for name in required_globals if name not in globals()]
assert not missing, f'Run the earlier notebook cells first. Missing: {missing}'

def archived_parity_row(example, current_record, current_hidden, archived_record, archived_hidden):
    current = np.asarray(current_hidden, dtype=np.float64)
    archived = np.asarray(archived_hidden, dtype=np.float64)
    assert current.ndim == 1 and np.isfinite(current).all(), 'Current model produced an invalid hidden state'
    current_scores = np.asarray([current_record['candidate_logprobs'][k] for k in example.labels], dtype=float)
    archived_scores = np.asarray([archived_record['candidate_logprobs'][k] for k in example.labels], dtype=float)
    assert np.isfinite(current_scores).all(), 'Current model produced invalid candidate scores'
    current_value = decode(current)
    assert np.isfinite(current_value), 'Current readout is non-finite'
    shape_ok = archived.shape == current.shape
    old_finite = bool(shape_ok and np.isfinite(archived).all() and np.isfinite(archived_scores).all())
    archived_value = decode(archived) if old_finite else None
    old_finite = old_finite and bool(np.isfinite(archived_value))
    # Use the same storage representation for diagnosis; do not assume FP16 caused a discrepancy.
    with np.errstate(over='ignore', invalid='ignore'):
        stored_current = current.astype(np.float16).astype(np.float32)
    cast_value = decode(stored_current) if np.isfinite(stored_current).all() else None
    gap = abs(current_value-archived_value) if old_finite else None
    score_gap = float(np.max(np.abs((current_scores-current_scores.mean())-
                                    (archived_scores-archived_scores.mean())))) if old_finite else None
    return dict(example_id=example.example_id, scene_id=str(example.scene_id), variant=example.variant,
        current_readout=float(current_value), archived_readout=float(archived_value) if old_finite else None,
        readout_abs_difference=float(gap) if gap is not None else None,
        centered_score_max_difference=score_gap,
        current_fp16_roundtrip_readout=float(cast_value) if cast_value is not None and np.isfinite(cast_value) else None,
        hidden_relative_l2=float(np.linalg.norm(current-archived)/max(np.linalg.norm(current),1e-12)) if old_finite else None,
        hidden_shape_matches=bool(shape_ok), archived_values_finite=bool(old_finite),
        parity_passed=bool(old_finite and gap <= .15 and score_gap <= .15))

def archive_reuse_decision(expected_ids, available_ids, parity_rows):
    complete = set(available_ids) == set(expected_ids) and bool(expected_ids)
    passed = bool(parity_rows) and all(r['parity_passed'] for r in parity_rows)
    if not complete:
        return 'fresh_capture', 'archive_missing_or_incomplete'
    if not passed:
        return 'fresh_capture', 'archive_numerical_parity_not_established'
    return 'verified_archive', 'complete_archive_and_audit_passed'

legacy_test = {}
legacy_test_provenance = None
legacy_config = CONFIRMATORY_ROOT/'capture_config.json'
legacy_shards = sorted(CONFIRMATORY_ROOT.glob('shard_*.pt'))
config = None
if legacy_config.is_file() and legacy_shards:
    config = read_json(legacy_config)['identity']
    # Identity/corruption errors remain errors. Only numerical non-equivalence changes the source.
    assert config['model_id'] == MODEL_ID and config['resolved_revision'] == MODEL_REVISION
    assert int(config['selected_layer_index']) == SELECTED_LAYER and float(config['frozen_ridge_alpha']) == FROZEN_ALPHA
    assert config['dtype'] == MODEL_DTYPE
    assert config['dataset_fingerprint'] == runtime_identity['dataset']
    assert config['dataset_scenes_sha256'] == sha256_file(DATASET_DIR/'scenes.jsonl')
    assert config['dataset_splits_sha256'] == sha256_file(DATASET_DIR/'splits.json')
    assert config['discovery_selection_sha256'] == sha256_file(discovery_selection_path)
    for path in tqdm(legacy_shards, desc='Reading original test activations'):
        payload = torch.load(path, map_location='cpu', weights_only=False)
        assert int(payload['selected_layer_index']) == SELECTED_LAYER
        assert len(payload['records']) == len(payload['activations']), f'Misaligned archive: {path}'
        for record, hidden in zip(payload['records'], payload['activations']):
            eid = record['example_id']
            assert eid in input_identities and eid not in legacy_test
            expected = input_identities[eid]
            for key in ('scene_id','variant','split','gold_semantic','label_to_semantic'):
                assert record[key] == expected[key], (eid,key)
            assert int(record['net_torque']) == expected['net_torque']
            legacy_test[eid] = (record, hidden.float().numpy())
    legacy_test_provenance = {str(p):sha256_file(p) for p in [legacy_config,*legacy_shards]}

# Freeze reuse once for these exact inputs/runtime/probe. The fix has its own code identity.
reuse_request = dict(schema='readout-test-reuse-2.0', implementation=TEST_REUSE_IMPLEMENTATION_SHA,
    protocol=PROTOCOL_SHA, runtime=RUNTIME_SHA, inputs=INPUT_SHA, probe=PROBE_SHA,
    numerical_lock=NUMERIC_LOCK, files=legacy_test_provenance,
    readout_parity_tolerance=.15, centered_score_parity_tolerance=.15,
    archive_policy='complete archive only; any failed audit means all 1007 scenes captured in current runtime')
reuse_directory = OUTPUT_ROOT/'test_reuse_v2'/identity_hash(reuse_request)
decision_path = reuse_directory/'decision.json'
if decision_path.is_file():
    reuse_payload = read_json(decision_path)
    assert reuse_payload['request'] == canonical(reuse_request), 'Reuse identity mismatch'
    parity_rows = reuse_payload['parity_rows']
else:
    parity_rows = []
    if legacy_test:
        # Same one-scene deterministic audit as before; no selection on correctness/effect size.
        audit_scene = min(test_by_scene, key=lambda s:stable_key('archive-parity:'+s))
        for ex in sorted(test_by_scene[audit_scene], key=lambda e:e.variant):
            if ex.example_id not in legacy_test:
                continue
            now, captured = forward_one(ex,(SELECTED_LAYER,))
            old, old_hidden = legacy_test[ex.example_id]
            parity_rows.append(archived_parity_row(ex,now,captured[SELECTED_LAYER],old,old_hidden))
    mode, reason = archive_reuse_decision(input_identities,legacy_test,parity_rows)
    reuse_payload = dict(request=canonical(reuse_request),mode=mode,reason=reason,
        parity_rows=parity_rows,expected_examples=len(input_identities),archived_examples=len(legacy_test))
    # Persist failure details before any long capture, including the original error's two readouts.
    write_json(decision_path,reuse_payload)
    if parity_rows:
        pd.DataFrame(parity_rows).to_csv(reuse_directory/'parity_diagnostics.csv',index=False)
mode, reason = archive_reuse_decision(input_identities,legacy_test,parity_rows)
assert (mode,reason) == (reuse_payload['mode'],reuse_payload['reason'])
TEST_REUSE_SHA = identity_hash(reuse_payload)
EVALUATION_STAGE = 'test_scene_shards_reuse_v2/' + TEST_REUSE_SHA
write_json(ANALYSIS_DIR/'test_capture_source.json',dict(mode=mode,reason=reason,
    decision_path=str(decision_path),reuse_sha=TEST_REUSE_SHA,probe_sha=PROBE_SHA,
    expected_scenes=len(test_by_scene),audit=parity_rows))
if parity_rows:
    display(pd.DataFrame(parity_rows)[['variant','current_readout','archived_readout',
        'readout_abs_difference','centered_score_max_difference','parity_passed']])
if mode == 'fresh_capture':
    legacy_test.clear()
    print('旧测试状态未获准复用：',reason)
    print('将使用当前模型与已冻结的读出器重新采集 1,007 个场景的六种映射，并按场景保存。')
else:
    print('旧测试状态通过核对，复用全部六映射归档。')
print('读出器、层号、0.05 修准标准均保持不变。诊断文件：',decision_path)

evaluation_rows = []
for sid in tqdm(sorted(test_by_scene),desc='1,007-scene readout/answer evaluation',unit='scene'):
    examples = sorted(test_by_scene[sid],key=lambda e:e.variant)
    def evaluate_scene(examples=examples):
        states,records = [],[]
        for e in examples:
            if mode == 'verified_archive':
                old,h = legacy_test[e.example_id]
                score = dict(predicted_semantic=old['predicted_semantic'],candidate_logprobs=old['candidate_logprobs'])
                provenance = 'historical_variant_activation'
            else:
                score,captured = forward_one(e,(SELECTED_LAYER,))
                h = captured[SELECTED_LAYER].astype(np.float16).astype(np.float32)
                provenance = 'new_capture'
            assert np.isfinite(h).all(), 'Test activation cannot be stored as float16'
            states.append(h)
            records.append(dict(example_id=e.example_id,variant=e.variant,provenance=provenance,
                decoded_torque=decode(h),predicted_semantic=score['predicted_semantic'],
                candidate_logprobs=score['candidate_logprobs'],correct=score['predicted_semantic']==str(e.gold_semantic)))
        scene_prediction = decode(np.mean(states,axis=0))
        fixed_variant = f"permutation_{int(stable_key('variant:'+str(examples[0].scene_id))[:8],16)%6}"
        fixed = next(r for r in records if r['variant']==fixed_variant)
        return dict(scene_id=str(examples[0].scene_id),gold_semantic=str(examples[0].gold_semantic),
            net_torque=int(examples[0].net_torque),scene_mean_prediction=scene_prediction,
            scene_mean_readout_correct=probe_outcome(scene_prediction)==str(examples[0].gold_semantic),
            six_map_model_accuracy=float(np.mean([r['correct'] for r in records])),
            fixed_variant=fixed_variant,fixed_readout_correct=probe_outcome(fixed['decoded_torque'])==str(examples[0].gold_semantic),
            fixed_model_correct=fixed['correct'],fixed_decoded_torque=fixed['decoded_torque'],variants=records,
            test_capture_mode=mode,test_reuse_sha=TEST_REUSE_SHA)
    evaluation_rows.append(cached_scene(EVALUATION_STAGE,sid,
        dict(probe=PROBE_SHA,inputs=INPUT_SHA,reuse=TEST_REUSE_SHA,numerical=NUMERIC_LOCK),evaluate_scene))
eval_frame = pd.DataFrame([{k:v for k,v in r.items() if k!='variants'} for r in evaluation_rows])
eval_frame.to_csv(ANALYSIS_DIR/'readout_1007_scenes.csv',index=False)
print('测试场景采集完成，可以继续运行后面的干预单元格。')


Run analysis:  73%|#######3  | 11/15 [00:00<?, ?cell/s]

旧测试状态未获准复用： archive_missing_or_incomplete
将使用当前模型与已冻结的读出器重新采集 1,007 个场景的六种映射，并按场景保存。
读出器、层号、0.05 修准标准均保持不变。诊断文件： /content/drive/MyDrive/VisualTorque/results/cross_model_readout_correction_v2/models/Qwen__Qwen3.5-27B__373612cb/test_reuse_v2/71e6eeb270df795cad6c5e8e97fc2daa214e6f633d790b301c9f4fa2d08010d7/decision.json


1,007-scene readout/answer evaluation:   0%|          | 0/1007 [00:00<?, ?scene/s]

测试场景采集完成，可以继续运行后面的干预单元格。


## 9. 304 场景的修准与对照，保留失败记录并按场景恢复


In [14]:
%%cell_progress 13/15 Run analysis
cell_stage('Running paired zero/oracle controls and preserving the original five doses')
# Original fixed-dose records are supplementary to the fresh zero/oracle pair.
legacy_fixed = {}
legacy_fixed_files = None
old_config_path = CAUSAL_ROOT/'capture_config.json'
old_manifest_path = CAUSAL_ROOT/'selection_manifest.jsonl'
old_shards = sorted((CAUSAL_ROOT/'shards').glob('batch_*.json'))
if not old_shards:
    old_shards = sorted((CAUSAL_ROOT/'intervention_shards').glob('batch_*.json'))
if old_config_path.is_file() and old_manifest_path.is_file() and old_shards:
    identity = read_json(old_config_path)['identity']
    assert identity['model_id'] == MODEL_ID and identity['resolved_revision'] == MODEL_REVISION
    assert identity['dtype'] == MODEL_DTYPE and identity['selected_layer_index'] == SELECTED_LAYER
    assert identity['dataset_fingerprint'] == runtime_identity['dataset']
    assert identity['discovery_selection_sha256'] == sha256_file(discovery_selection_path)
    assert identity['frozen_alpha'] == FROZEN_ALPHA
    assert abs(identity['torque_shift_unit']-TORQUE_SHIFT_UNIT) < 1e-6
    assert identity['discovery_activations_sha256'] == sha256_file(discovery_activation_path)
    assert identity['selection_manifest_sha256'] == sha256_file(old_manifest_path)
    assert read_jsonl(old_manifest_path) == selection_manifest
    for path in old_shards:
        shard = read_json(path)
        assert shard['capture_identity_sha256'] == sha256_file(old_config_path)
        for r in shard['records']:
            if r['condition_family'] != 'torque_direction':
                continue
            key = (r['example_id'], float(r['dose_multiplier']))
            assert key not in legacy_fixed
            legacy_fixed[key] = r
    legacy_fixed_files = {str(p):sha256_file(p) for p in [old_config_path,old_manifest_path,*old_shards]}
FIXED_REUSE_SHA = lock_json(OUTPUT_ROOT/'fixed_dose_reuse_identity.json',dict(files=legacy_fixed_files))
all_records = []
for example in tqdm(selected_examples, desc='Numerical correction', unit='scene'):
    sid = str(example.scene_id)
    def run_scene(example=example, sid=sid):
        zero, _ = forward_one(example, (SELECTED_LAYER,), zero_edit)
        oracle, _ = forward_one(example, (SELECTED_LAYER,), oracle_edit(example.net_torque))
        rows = [zero,oracle]
        if sid in control_ids:
            for kind in ('random','orthogonal'):
                for seed in RANDOM_SEEDS:
                    row, _ = forward_one(example, (SELECTED_LAYER,), control_edit(sid,kind,seed,oracle['actual_delta_norm']))
                    rows.append(row)
        complete_legacy = all((example.example_id,m) in legacy_fixed for m in DOSE_MULTIPLIERS)
        for multiplier in DOSE_MULTIPLIERS:
            if complete_legacy:
                r = legacy_fixed[(example.example_id,multiplier)]
                old_zero = legacy_fixed[(example.example_id,0.)]
                row = dict(example_id=example.example_id,scene_id=sid,variant=example.variant,
                    split='probe_test',problem_type=example.problem_type,net_torque=int(example.net_torque),
                    gold_semantic=str(example.gold_semantic),predicted_semantic=r['predicted_semantic'],correct=r['correct'],
                    condition=f'fixed_{multiplier:+g}',dose_multiplier=multiplier,
                    semantic_logprobs={k:r['logprob_'+k] for k in ('left','right','balanced')},
                    decoded_pre=r['decoded_torque_pre'],decoded_post=r['decoded_torque_post'],
                    actual_delta_norm=r['realized_delta_norm'], provenance='historical_v2',
                    fixed_baseline_correct=old_zero['correct'])
            else:
                row, _ = forward_one(example,(SELECTED_LAYER,),fixed_edit(multiplier))
                row.update(provenance='new_v2_rule',fixed_baseline_correct=zero['correct'])
            rows.append(row)
        for r in rows:
            r.setdefault('provenance','new_capture')
            r['original_model_correct'] = zero['correct']
            r['original_readout_correct'] = probe_outcome(zero['decoded_pre']) == str(example.gold_semantic)
            r['cohort_sha'] = COHORT_SHA
            r['model_id'],r['revision'],r['layer'] = MODEL_ID,MODEL_REVISION,SELECTED_LAYER
        return rows
    rows = cached_scene('intervention_scene_shards',sid,
        dict(probe=PROBE_SHA, input=input_identities[example.example_id], cohort=COHORT_SHA,
             numerical=NUMERIC_LOCK, fixed_reuse=FIXED_REUSE_SHA),run_scene)
    assert len(rows) == (15 if sid in control_ids else 7)
    all_records.extend(rows)
assert len(all_records) == 304*7+72*8
write_json(ANALYSIS_DIR/'intervention_records.json',all_records)
record_frame = pd.DataFrame(all_records)
record_frame.to_csv(ANALYSIS_DIR/'intervention_records.csv',index=False)


Run analysis:  80%|########  | 12/15 [00:00<?, ?cell/s]

Numerical correction:   0%|          | 0/304 [00:00<?, ?scene/s]

## 10. 汇总读出、答案、修正与新增错误，报告区间


In [15]:
%%cell_progress 14/15 Run analysis
cell_stage('Writing readable numerical and behavioral summaries with separate denominators')
record_frame['probe_correct'] = [probe_outcome(float(r['decoded_post']))==r['gold_semantic'] for r in all_records]
record_frame['absolute_readout_error'] = (record_frame['decoded_post']-record_frame['net_torque']).abs()
record_frame['model_change'] = record_frame['correct'].astype(float)-record_frame['original_model_correct'].astype(float)
record_frame['correct_side_margin'] = [
    (r['semantic_logprobs']['left']-r['semantic_logprobs']['right']) * (1 if r['net_torque']>0 else -1)
    if r['net_torque'] != 0 else np.nan for r in all_records]
summary_rows = []
for group_name, mask in [('all_304',np.ones(len(record_frame),dtype=bool)),
                         ('tilt_263',record_frame['net_torque'].ne(0)),
                         ('balance_41',record_frame['net_torque'].eq(0))]:
    for condition, frame in record_frame.loc[mask].groupby('condition'):
        # Seeds are repeated measurements: average inside scene, never inflate n.
        per_scene = frame.groupby('scene_id')[['correct','probe_correct','absolute_readout_error','model_change']].mean()
        accuracy_ci = scene_bootstrap(per_scene['correct'].to_numpy())
        change_ci = scene_bootstrap(per_scene['model_change'].to_numpy())
        if condition.startswith('fixed_'):
            fixed_change = frame['correct'].astype(float)-frame['fixed_baseline_correct'].astype(float)
            change_ci = scene_bootstrap(fixed_change.to_numpy())
        summary_rows.append(dict(group=group_name,condition=condition,scenes=len(per_scene),
            readout_mae=float(per_scene['absolute_readout_error'].mean()),
            readout_accuracy=float(per_scene['probe_correct'].mean()),
            model_accuracy=accuracy_ci['mean'],accuracy_lower=accuracy_ci['lower'],accuracy_upper=accuracy_ci['upper'],
            model_change=change_ci['mean'],change_lower=change_ci['lower'],change_upper=change_ci['upper']))
summary = pd.DataFrame(summary_rows)
summary.to_csv(ANALYSIS_DIR/'summary_by_cohort.csv',index=False)
zero = record_frame[record_frame['condition']=='zero'].copy()
oracle = record_frame[record_frame['condition']=='oracle'].copy()
assert zero['scene_id'].is_unique and oracle['scene_id'].is_unique
assert set(zero['scene_id']) == set(oracle['scene_id'])
achieved = oracle['achieved'].eq(True)
paired_effect = scene_bootstrap(oracle['model_change'].to_numpy())
repaired = int((~oracle['original_model_correct'].astype(bool)&oracle['correct'].astype(bool)).sum())
harmed = int((oracle['original_model_correct'].astype(bool)&~oracle['correct'].astype(bool)).sum())
original_errors = int((~oracle['original_model_correct'].astype(bool)).sum())
original_correct = len(oracle)-original_errors
technical = dict(oracle_achieved=int(achieved.sum()),oracle_total=len(oracle),
    oracle_failed=int((~achieved).sum()), zero_or_tiny=int((oracle['actual_delta_norm']<=1e-10).sum()),
    repaired=repaired,original_errors=original_errors,harmed=harmed,original_correct=original_correct,
    readout_correct_model_wrong=int((zero['original_readout_correct'].astype(bool)&~zero['correct'].astype(bool)).sum()),
    all_scene_accuracy_change=paired_effect,
    achieved_subset_accuracy_change=scene_bootstrap(oracle.loc[achieved,'model_change'].to_numpy()) if achieved.any() else None)
oracle.loc[~achieved].to_csv(ANALYSIS_DIR/'oracle_write_failures.csv',index=False)
controls = record_frame[record_frame['condition'].isin(['random','orthogonal'])].copy()
controls['valid_control'] = controls['norm_matched'].eq(True) & (
    controls['condition'].ne('orthogonal') | controls['orthogonal_preserved'].eq(True))
controls.loc[~controls['valid_control']].to_csv(ANALYSIS_DIR/'control_match_failures.csv',index=False)
control_contrasts = []
for kind, frame in controls.groupby('condition'):
    counts = frame.groupby('scene_id').size()
    assert len(counts) == 72 and set(counts) == {4}
    per_scene = frame.groupby('scene_id')['model_change'].mean().reset_index()
    matched_oracle = oracle[oracle['scene_id'].isin(per_scene['scene_id'])]
    difference = paired_on_ids(matched_oracle,per_scene,'scene_id','model_change')
    control_contrasts.append(dict(control=kind,scope='all_72_attempted',
        **scene_bootstrap(difference),invalid_records=int((~frame['valid_control']).sum())))
    valid_ids = set(frame.groupby('scene_id')['valid_control'].all().loc[lambda x:x].index)
    valid_ids &= set(oracle.loc[achieved,'scene_id'])
    if valid_ids:
        difference = paired_on_ids(matched_oracle[matched_oracle['scene_id'].isin(valid_ids)],
            per_scene[per_scene['scene_id'].isin(valid_ids)],'scene_id','model_change')
        control_contrasts.append(dict(control=kind,scope='oracle_achieved_all_four_controls_valid',**scene_bootstrap(difference)))
pd.DataFrame(control_contrasts).to_csv(ANALYSIS_DIR/'paired_control_accuracy.csv',index=False)
# Margin contrasts are defined only for the original 263 nonbalanced scenes.
tilt_zero = zero[zero['net_torque'].ne(0)][['scene_id','correct_side_margin']]
margin_rows = []
for condition, frame in record_frame[record_frame['net_torque'].ne(0)&~record_frame['condition'].str.startswith('fixed_')].groupby('condition'):
    per_scene = frame.groupby('scene_id')['correct_side_margin'].mean().reset_index()
    diff = paired_on_ids(per_scene,tilt_zero[tilt_zero['scene_id'].isin(per_scene['scene_id'])],'scene_id','correct_side_margin')
    margin_rows.append(dict(condition=condition,**scene_bootstrap(diff)))
pd.DataFrame(margin_rows).to_csv(ANALYSIS_DIR/'tilt_margin_changes.csv',index=False)
for key, frame in [('readout_correct',oracle[oracle['original_readout_correct'].eq(True)]),
                    ('readout_wrong',oracle[oracle['original_readout_correct'].eq(False)])]:
    if len(frame):
        technical[key] = dict(scenes=len(frame),**{'accuracy_change':scene_bootstrap(frame['model_change'].to_numpy())})
evaluation_summary = dict(scenes=len(eval_frame),
    six_map_scene_mean_mae=float(np.abs(eval_frame['scene_mean_prediction']-eval_frame['net_torque']).mean()),
    six_map_scene_mean_readout_accuracy=float(eval_frame['scene_mean_readout_correct'].mean()),
    six_map_model_accuracy=float(eval_frame['six_map_model_accuracy'].mean()),
    single_mapping_readout_accuracy=float(eval_frame['fixed_readout_correct'].mean()),
    single_mapping_model_accuracy=float(eval_frame['fixed_model_correct'].mean()),
    single_mapping_readout_correct_model_wrong=int((eval_frame['fixed_readout_correct']&~eval_frame['fixed_model_correct']).sum()))
write_json(ANALYSIS_DIR/'result_summary.json',dict(model_key=MODEL_KEY,model_id=MODEL_ID,revision=MODEL_REVISION,
    layer=SELECTED_LAYER,probe_sha=PROBE_SHA,cohort_sha=COHORT_SHA,technical=technical,evaluation=evaluation_summary,
    interpretation='Oracle correction edits an external readout; it does not establish repair of native physical computation.'))
display(summary[summary['condition'].isin(['zero','oracle'])])
display(pd.DataFrame([technical]))
lines = ['# 净力矩读出与数值修准实验', '', f'模型：{MODEL_ID}；读出层：{SELECTED_LAYER}。', '',
    f'304 个场景中，原读出正确但模型答错的有 {technical["readout_correct_model_wrong"]} 个。',
    f'数值写入达到预定误差要求的有 {int(achieved.sum())}/304 个；其余记录单独保留。',
    f'在全部 304 个场景上，修准操作修正了 {repaired}/{original_errors} 个原有错误，并使 {harmed}/{original_correct} 个原来正确的答案变错。',
    f'模型准确率变化：{paired_effect["mean"]*100:.2f} 个百分点；95% 场景配对区间为 [{paired_effect["lower"]*100:.2f}, {paired_effect["upper"]*100:.2f}]。',
    '', '随机与正交对照按场景平均四个方向；写入失败与范数不匹配记录见单独表格。',
    '本轮使用原来的物理场景，是配对新增诊断；不构成新家族独立复现。',
    '修准外部预测器不等于修复模型原生物理计算。是否支持较小的答案效应，需要结合区间与成功写入比例判断。']
(ANALYSIS_DIR/'REPORT_ZH.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')


Run analysis:  87%|########6 | 13/15 [00:00<?, ?cell/s]

,group,condition,scenes,readout_mae,readout_accuracy,model_accuracy,accuracy_lower,accuracy_upper,model_change,change_lower,change_upper
5,all_304,oracle,304,0.026132,1.000000,0.453947,0.398026,0.509868,0.003289,0.0,0.009868
8,all_304,zero,304,3.463861,0.779605,0.450658,0.394737,0.506579,0.000000,0.0,0.000000
14,tilt_263,oracle,263,0.026438,1.000000,0.494297,0.433460,0.555133,0.003802,0.0,0.011407
17,tilt_263,zero,263,3.494232,0.889734,0.490494,0.429658,0.551331,0.000000,0.0,0.000000
23,balance_41,oracle,41,0.024171,1.000000,0.195122,0.073171,0.317073,0.000000,0.0,0.000000
26,balance_41,zero,41,3.269038,0.073171,0.195122,0.073171,0.317073,0.000000,0.0,0.000000


,oracle_achieved,oracle_total,oracle_failed,zero_or_tiny,repaired,original_errors,harmed,original_correct,readout_correct_model_wrong,all_scene_accuracy_change,achieved_subset_accuracy_change,readout_correct,readout_wrong
0,302,304,2,2,1,167,0,137,117,"{'mean': 0.003289473684210526, 'n': 304, 'lowe...","{'mean': 0.0033112582781456954, 'n': 302, 'low...","{'scenes': 237, 'accuracy_change': {'mean': 0....","{'scenes': 67, 'accuracy_change': {'mean': 0.0..."


326

326

## 11. 三模型共同场景比较：不把显著与不显著直接作差异结论


In [16]:
%%cell_progress 15/15 Run analysis
cell_stage('Comparing paired effects only on identical scene manifests')
import itertools
coverage, effects = [], {}
for key, spec in MODEL_SPECS.items():
    directory = SHARED_ROOT/'models'/spec[1]/'analysis'
    summary_path, raw_path = directory/'result_summary.json', directory/'intervention_records.json'
    if not summary_path.is_file() or not raw_path.is_file():
        coverage.append(dict(model_key=key,status='not_complete'))
        continue
    result = read_json(summary_path)
    assert result['cohort_sha'] == COHORT_SHA
    coverage.append(dict(model_key=key,status='complete',**result['evaluation']))
    raw = read_json(raw_path)
    frame = pd.DataFrame([r for r in raw if r['condition']=='oracle'])
    assert len(frame)==304 and frame['scene_id'].is_unique
    frame['model_change'] = frame['correct'].astype(float)-frame['original_model_correct'].astype(float)
    effects[key] = frame[['scene_id','model_change']]
paired_rows = []
for left,right in itertools.combinations(effects,2):
    differences = paired_on_ids(effects[left],effects[right],'scene_id','model_change')
    paired_rows.append(dict(left_model=left,right_model=right,**scene_bootstrap(differences)))
cross_dir = SHARED_ROOT/'cross_model_analysis'
pd.DataFrame(coverage).to_csv(cross_dir/'coverage_and_readout.csv',index=False)
pd.DataFrame(paired_rows).to_csv(cross_dir/'paired_accuracy_differences.csv',index=False)
display(pd.DataFrame(coverage))
display(pd.DataFrame(paired_rows))
print('Outputs:', ANALYSIS_DIR)
print('Pairwise intervals are descriptive; no significance claims from multiple comparisons are auto-generated.')


Run analysis:  93%|#########3| 14/15 [00:00<?, ?cell/s]

,model_key,status,scenes,single_mapping_model_accuracy,single_mapping_readout_accuracy,single_mapping_readout_correct_model_wrong,six_map_model_accuracy,six_map_scene_mean_mae,six_map_scene_mean_readout_accuracy
0,qwen9b,complete,1007,0.613704,0.850050,301,0.599470,3.648875,0.857001
1,gemma12b,complete,1007,0.711023,0.867925,238,0.710526,2.852465,0.883813
2,qwen27b,complete,1007,0.545184,0.853029,357,0.549156,3.452468,0.865938


,left_model,right_model,mean,n,lower,upper
0,qwen9b,gemma12b,0.006579,304,-0.006579,0.019737
1,qwen9b,qwen27b,0.000000,304,-0.013158,0.013158
2,gemma12b,qwen27b,-0.006579,304,-0.016447,0.000000


Outputs: /content/drive/MyDrive/VisualTorque/results/cross_model_readout_correction_v2/models/Qwen__Qwen3.5-27B__373612cb/analysis
Pairwise intervals are descriptive; no significance claims from multiple comparisons are auto-generated.
